In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, subprocess, os
z = glob.glob("/content/drive/MyDrive/**/drift-conference*.zip", recursive=True)[0]
print("zip:", z)
subprocess.run(["rm","-rf","/content/dc_pkg"], check=True)
os.makedirs("/content/dc_pkg", exist_ok=True)
subprocess.run(["unzip","-q",z,"-d","/content/dc_pkg"], check=True)
subprocess.run(["pip","-q","install","--no-deps","/content/dc_pkg/drift-conference"], check=True)

import driftbench as db
print("driftbench:", db.__version__)            # expect 0.1.1
from driftbench.cleaning import clean_csv_streaming
print("OK")

In [ ]:
print(open("/content/dc_pkg/drift-conference/pyproject.toml").read().split("version =")[1][:12])
print("clean_csv_streaming" in open("/content/dc_pkg/drift-conference/src/driftbench/cleaning.py").read())

In [ ]:
import psutil, os
m = psutil.virtual_memory()
print(f"RAM used {m.percent}%  ({m.used/1e9:.1f} / {m.total/1e9:.1f} GB)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, subprocess, os, sys
z = glob.glob("/content/drive/MyDrive/**/drift-conference*.zip", recursive=True)[0]
subprocess.run(["rm","-rf","/content/dc_pkg"], check=True)
os.makedirs("/content/dc_pkg", exist_ok=True)
subprocess.run(["unzip","-q",z,"-d","/content/dc_pkg"], check=True)
subprocess.run(["pip","-q","install","--no-deps","/content/dc_pkg/drift-conference"], check=True)
for m in list(sys.modules):
    if m=="driftbench" or m.startswith("driftbench."): del sys.modules[m]
from driftbench import config, random_split, compute_metrics, metrics_frame
print("ready — config + models imported")

In [ ]:
import pandas as pd, numpy as np, gc, warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, matthews_corrcoef
from driftbench import config, random_split
warnings.filterwarnings('ignore')
R = Path('/content/drive/MyDrive/drift-conference/results')
rng = np.random.default_rng(0)

def ci(y_true, y_pred, fn, B=1000, cap=100_000):
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); n=len(y_true)
    if n>cap:                                   # subsample once for the bootstrap
        s=rng.choice(n,cap,replace=False); y_true,y_pred=y_true[s],y_pred[s]; n=cap
    pt=fn(y_true,y_pred)
    bs=np.array([fn(y_true[i],y_pred[i]) for i in (rng.integers(0,n,n) for _ in range(B))])
    return pt, np.percentile(bs,2.5), np.percentile(bs,97.5)
macro=lambda a,b: f1_score(a,b,average='macro',zero_division=0)

def rf(): return RandomForestClassifier(n_estimators=100,max_depth=20,n_jobs=-1,random_state=0,class_weight='balanced_subsample')

p=config.INTERIM_DIR/'CSE-CIC-IDS2018'
X=pd.read_parquet(p/'features.parquet').astype('float32')
y=pd.read_parquet(p/'labels.parquet')['label'].reset_index(drop=True)
ts=pd.to_datetime(pd.read_parquet(p/'metadata.parquet')['timestamp'],dayfirst=True,errors='coerce')
order=np.argsort(ts.values,kind='stable'); a=int(len(X)*0.6)

out=[]
rs=random_split(len(X),y,seed=0); m=rf().fit(X.iloc[rs['train']],y.iloc[rs['train']])
out.append(('demoA_random_macroF1',)+ci(y.iloc[rs['test']],m.predict(X.iloc[rs['test']]),macro)); del m; gc.collect()
m=rf().fit(X.iloc[order[:a]],y.iloc[order[:a]])
out.append(('demoA_blind_macroF1',)+ci(y.iloc[order[a:]],m.predict(X.iloc[order[a:]]),macro)); del m; gc.collect()

Xa=pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'features_core.parquet').astype('float32')
ya=pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
Xb=pd.read_parquet(config.PROCESSED_DIR/'CIC-DDoS2019'/'features_core.parquet').astype('float32')
yb=pd.read_parquet(config.PROCESSED_DIR/'CIC-DDoS2019'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
sh=['benign','ddos']; ma=ya.isin(sh).values; mb=yb.isin(sh).values
Xa,ya=Xa[ma],ya[ma]; Xb,yb=Xb[mb].reset_index(drop=True)[Xa.columns],yb[mb].reset_index(drop=True)
m=rf().fit(Xa,ya)
out.append(('demoB_zeroshot_mcc',)+ci(yb,m.predict(Xb),matthews_corrcoef)); del m; gc.collect()

res=pd.DataFrame(out,columns=['metric','point','ci_low','ci_high'])
res.to_csv(R/'bootstrap_cis.csv',index=False)
print(res.round(4).to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path
R = Path('/content/drive/MyDrive/drift-conference/results')
print("bootstrap saved:", (R/'bootstrap_cis.csv').exists())

In [ ]:
import pandas as pd, numpy as np, gc, warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from driftbench import config, random_split
warnings.filterwarnings('ignore')
R = Path('/content/drive/MyDrive/drift-conference/results')
rng = np.random.default_rng(0)
macro = lambda a,b: f1_score(a,b,average='macro',zero_division=0)
def rf(): return RandomForestClassifier(n_estimators=100,max_depth=20,n_jobs=-1,random_state=0,class_weight='balanced_subsample')

p=config.INTERIM_DIR/'CSE-CIC-IDS2018'
X=pd.read_parquet(p/'features.parquet').astype('float32')
y=pd.read_parquet(p/'labels.parquet')['label'].reset_index(drop=True)
ts=pd.to_datetime(pd.read_parquet(p/'metadata.parquet')['timestamp'],dayfirst=True,errors='coerce')
order=np.argsort(ts.values,kind='stable'); a=int(len(X)*0.6)

# fit both conditions, get per-test predictions
rs=random_split(len(X),y,seed=0)
m=rf().fit(X.iloc[rs['train']],y.iloc[rs['train']]); yr_true=y.iloc[rs['test']].to_numpy(); yr_pred=m.predict(X.iloc[rs['test']]); del m; gc.collect()
m=rf().fit(X.iloc[order[:a]],y.iloc[order[:a]]); yb_true=y.iloc[order[a:]].to_numpy(); yb_pred=m.predict(X.iloc[order[a:]]); del m; gc.collect()

# ---- (1) paired bootstrap significance test on the gap (random - blind) ----
B=1000; cap=100_000
def sub(t,pp):
    n=len(t)
    if n>cap: s=rng.choice(n,cap,replace=False); return t[s],pp[s]
    return t,pp
rt,rp=sub(yr_true,yr_pred); bt,bp=sub(yb_true,yb_pred)
gaps=[]
for _ in range(B):
    i=rng.integers(0,len(rt),len(rt)); j=rng.integers(0,len(bt),len(bt))
    gaps.append(macro(rt[i],rp[i]) - macro(bt[j],bp[j]))
gaps=np.array(gaps)
obs_gap=macro(yr_true,yr_pred)-macro(yb_true,yb_pred)
p_emp=(gaps<=0).mean()                      # fraction where gap reverses/vanishes
print(f"[Significance] observed random−blind macro-F1 gap = {obs_gap:.4f}")
print(f"  bootstrap gap 95% CI = [{np.percentile(gaps,2.5):.4f}, {np.percentile(gaps,97.5):.4f}]")
print(f"  empirical p-value (gap<=0) = {p_emp:.4f}  (<0.001 if 0/{B})")

# ---- (2) label-shuffle control: train on permuted labels (random split) ----
y_perm = pd.Series(rng.permutation(y.iloc[rs['train']].to_numpy()), index=rs['train'])
m=rf().fit(X.iloc[rs['train']], y_perm); shuf=macro(yr_true, m.predict(X.iloc[rs['test']])); del m; gc.collect()
chance = 1.0/ y.nunique()
print(f"\n[Label-shuffle] macro-F1 with permuted training labels = {shuf:.4f}")
print(f"  (real random-split macro-F1 = {macro(yr_true,yr_pred):.4f}; naive chance ~ {chance:.4f})")

pd.DataFrame([{'observed_gap':obs_gap,'gap_ci_low':np.percentile(gaps,2.5),'gap_ci_high':np.percentile(gaps,97.5),
               'p_value':p_emp,'shuffle_macroF1':shuf,'real_macroF1':macro(yr_true,yr_pred)}]
            ).to_csv(R/'controls.csv',index=False)
print("\nsaved controls.csv")

In [ ]:
import glob
from pathlib import Path
test_files = sorted(glob.glob('/content/ddos_tmp/03-11/*.csv'))
print("03-11 files:", len(test_files))
for f in test_files: print(" ", Path(f).name)

In [ ]:
!pip -q install kaggle
from google.colab import files
files.upload()                      # kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!mkdir -p /content/ddos_tmp
!kaggle datasets download -d rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files -p /content/ddos_tmp --unzip
import glob
from pathlib import Path
test_files = sorted(glob.glob('/content/ddos_tmp/03-11/*.csv'))
print("03-11 files:", len(test_files))
for f in test_files: print(" ", Path(f).name)

In [ ]:
import numpy as np, pandas as pd
from collections import Counter
from driftbench.cleaning import clean_flows

def clean_csv_streaming(csv_paths, cap_per_class=200_000, chunksize=500_000, seed=0, dedup=True):
    rng = np.random.default_rng(seed); counts = Counter()
    for p in csv_paths:
        hdr = pd.read_csv(p, nrows=0)
        lc=[c for c in hdr.columns if str(c).strip().lower()=="label"]; lc=lc[0] if lc else hdr.columns[-1]
        for ch in pd.read_csv(p, usecols=[lc], chunksize=2_000_000, low_memory=False):
            counts.update(ch[lc].astype(str).str.strip().value_counts().to_dict())
    keep={c:min(1.0,cap_per_class/n) for c,n in counts.items()} if cap_per_class else None
    F,M,L,dd=[],[],[],0
    for p in csv_paths:
        for ch in pd.read_csv(p, chunksize=chunksize, low_memory=False):
            r=clean_flows(ch, drop_constant=False, dedup=dedup); dd+=r.n_dedup_removed
            f,m,l=r.features,r.metadata,r.labels
            if keep is not None:
                pr=l.map(lambda c: keep.get(c,1.0)).to_numpy(float); mask=rng.random(len(pr))<pr
                f,m,l=f[mask],m[mask],l[mask]
            if len(l): F.append(f);M.append(m);L.append(l)
    feats=pd.concat(F,ignore_index=True); meta=pd.concat(M,ignore_index=True)
    labs=pd.concat(L,ignore_index=True).reset_index(drop=True)
    feats=feats.fillna(feats.median(numeric_only=True))
    const=feats.nunique(); dropped=const[const<=1].index.tolist(); feats=feats.drop(columns=dropped)
    class R: pass
    r=R(); r.features=feats;r.metadata=meta;r.labels=labs;r.dropped_constant=dropped;r.n_dedup_removed=dd
    return r

from pathlib import Path
from driftbench import config
test_files = sorted(Path('/content/ddos_tmp/03-11').glob('*.csv'))
res = clean_csv_streaming(test_files, cap_per_class=200_000, chunksize=500_000, seed=0)
print("03-11 features", res.features.shape, "| dropped_const", len(res.dropped_constant))
print("rows per class:", res.labels.value_counts().to_dict())

out = config.INTERIM_DIR/'CIC-DDoS2019-test'; out.mkdir(parents=True, exist_ok=True)
res.features.to_parquet(out/'features.parquet')
res.metadata.to_parquet(out/'metadata.parquet')
res.labels.to_frame('label').to_parquet(out/'labels.parquet')
print("written:", out)

In [ ]:
import pandas as pd, numpy as np, re, gc, warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from driftbench import config, compute_metrics, metrics_frame, harmonise_labels
warnings.filterwarnings('ignore')
R = Path('/content/drive/MyDrive/drift-conference/results')

def canon(col):
    c=col.strip().lower(); c=re.sub(r'[\s]+','_',c)
    for pat,sub in [(r'\bpkt\b','packet'),(r'\bpkts\b','packets'),(r'\blen\b','length'),
        (r'\bbyts\b','bytes'),(r'\bcnt\b','count'),(r'\btot\b','total'),
        (r'segment_size','seg_size'),(r'seg_size_avg','seg_size_mean'),
        (r'avg_(fwd|bwd)_seg_size',r'\1_seg_size_mean'),
        (r'init_win_bytes_forward','init_fwd_win_bytes'),(r'init_win_bytes_backward','init_bwd_win_bytes'),
        (r'act_data_pkt_fwd','fwd_act_data_packets'),(r'fwd_act_data_pkts','fwd_act_data_packets'),
        (r'_byts/s','_bytes/s'),(r'flow_byts/s','flow_bytes/s'),(r'flow_pkts/s','flow_packets/s'),
        (r'\bpacket_length\b','packet_len')]:
        c=re.sub(pat,sub,c)
    return c.replace('packet','pkt').replace('length','len')

core = pd.read_csv(config.PROCESSED_DIR/'common_core_features.csv').iloc[:,0].tolist()

# build harmonised 45-core view for the new test regime
pt = config.INTERIM_DIR/'CIC-DDoS2019-test'
Xt = pd.read_parquet(pt/'features.parquet'); Xt = Xt.rename(columns={c:canon(c) for c in Xt.columns})
Xt = Xt.loc[:, [c for c in core if c in Xt.columns]].loc[:, lambda d: ~d.columns.duplicated()].astype('float32')
yt = harmonise_labels(pd.read_parquet(pt/'labels.parquet')['label']).astype(str).reset_index(drop=True)
print("regime2 core:", Xt.shape, "| families:", yt.value_counts().to_dict())

# train on 2018 shared families (same as Demo B)
Xa = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'features_core.parquet').astype('float32')
ya = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
sh=['benign','ddos']; ma=ya.isin(sh).values
Xa,ya = Xa[ma], ya[ma]
mt = yt.isin(sh).values
Xt,yt = Xt[mt].reset_index(drop=True)[Xa.columns], yt[mt].reset_index(drop=True)
print("train(2018):", Xa.shape, "| test(2019-03-11):", Xt.shape, dict(yt.value_counts()))

base = RandomForestClassifier(n_estimators=120,max_depth=24,n_jobs=-1,random_state=0,class_weight='balanced_subsample').fit(Xa,ya)
rows={}
rows['zero_shot'] = compute_metrics(yt, base.predict(Xt), base.predict_proba(Xt), base.classes_)
order = np.arange(len(Xt)); rng=np.random.default_rng(0); rng.shuffle(order)   # no usable ts; random buffer
for frac in (0.01,0.05,0.10):
    k=max(int(len(order)*frac),1); buf,rest=order[:k],order[k:]
    try:
        from sklearn.frozen import FrozenEstimator
        cal=CalibratedClassifierCV(FrozenEstimator(base),method='sigmoid')
    except ImportError:
        cal=CalibratedClassifierCV(base,method='sigmoid',cv='prefit')
    cal.fit(Xt.iloc[buf],yt.iloc[buf])
    rows[f'buffer_{int(frac*100)}pct']=compute_metrics(yt.iloc[rest],cal.predict(Xt.iloc[rest]),cal.predict_proba(Xt.iloc[rest]),cal.classes_)
    gc.collect()

tbl=metrics_frame(rows); tbl.to_csv(R/'demoB_transfer_regime2_rf.csv')
print("\n=== Demo B regime 2: 2018 -> 2019(03-11) ===")
print(tbl[['macro_f1','weighted_f1','mcc','brier','ece']].round(4))

In [ ]:
yt_raw = pd.read_parquet(config.INTERIM_DIR/'CIC-DDoS2019-test'/'labels.parquet')['label']
yt = harmonise_labels(yt_raw).astype(str).reset_index(drop=True)
yt = yt.mask(yt_raw.str.strip().str.lower().str.contains('portmap'), 'ddos').reset_index(drop=True)
print("after Portmap fix:", yt.value_counts().to_dict())

In [ ]:
import numpy as np, gc, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from driftbench import config, compute_metrics, metrics_frame
import pandas as pd, re
from pathlib import Path
warnings.filterwarnings('ignore')
R = Path('/content/drive/MyDrive/drift-conference/results')

def canon(col):
    c=col.strip().lower(); c=re.sub(r'[\s]+','_',c)
    for pat,sub in [(r'\bpkt\b','packet'),(r'\bpkts\b','packets'),(r'\blen\b','length'),
        (r'\bbyts\b','bytes'),(r'\bcnt\b','count'),(r'\btot\b','total'),
        (r'segment_size','seg_size'),(r'seg_size_avg','seg_size_mean'),
        (r'avg_(fwd|bwd)_seg_size',r'\1_seg_size_mean'),
        (r'init_win_bytes_forward','init_fwd_win_bytes'),(r'init_win_bytes_backward','init_bwd_win_bytes'),
        (r'act_data_pkt_fwd','fwd_act_data_packets'),(r'fwd_act_data_pkts','fwd_act_data_packets'),
        (r'_byts/s','_bytes/s'),(r'flow_byts/s','flow_bytes/s'),(r'flow_pkts/s','flow_packets/s'),
        (r'\bpacket_length\b','packet_len')]:
        c=re.sub(pat,sub,c)
    return c.replace('packet','pkt').replace('length','len')
core = pd.read_csv(config.PROCESSED_DIR/'common_core_features.csv').iloc[:,0].tolist()

# rebuild regime-2 features aligned to 45 core (yt already fixed in memory)
Xt = pd.read_parquet(config.INTERIM_DIR/'CIC-DDoS2019-test'/'features.parquet')
Xt = Xt.rename(columns={c:canon(c) for c in Xt.columns})
Xt = Xt.loc[:, [c for c in core if c in Xt.columns]].loc[:, lambda d: ~d.columns.duplicated()].astype('float32')

Xa = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'features_core.parquet').astype('float32')
ya = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
sh=['benign','ddos']; ma=ya.isin(sh).values; Xa,ya=Xa[ma],ya[ma]
mt = yt.isin(sh).values
Xt2,yt2 = Xt[mt].reset_index(drop=True)[Xa.columns], yt[mt].reset_index(drop=True)
print("train(2018):", Xa.shape, "| test(2019-03-11, Portmap=ddos):", Xt2.shape, dict(yt2.value_counts()))

base = RandomForestClassifier(n_estimators=120,max_depth=24,n_jobs=-1,random_state=0,class_weight='balanced_subsample').fit(Xa,ya)
rows={}
rows['zero_shot']=compute_metrics(yt2, base.predict(Xt2), base.predict_proba(Xt2), base.classes_)
order=np.arange(len(Xt2)); rng=np.random.default_rng(0); rng.shuffle(order)
for frac in (0.01,0.05,0.10):
    k=max(int(len(order)*frac),1); buf,rest=order[:k],order[k:]
    try:
        from sklearn.frozen import FrozenEstimator
        cal=CalibratedClassifierCV(FrozenEstimator(base),method='sigmoid')
    except ImportError:
        cal=CalibratedClassifierCV(base,method='sigmoid',cv='prefit')
    cal.fit(Xt2.iloc[buf],yt2.iloc[buf])
    rows[f'buffer_{int(frac*100)}pct']=compute_metrics(yt2.iloc[rest],cal.predict(Xt2.iloc[rest]),cal.predict_proba(Xt2.iloc[rest]),cal.classes_)
    gc.collect()

tbl=metrics_frame(rows); tbl.to_csv(R/'demoB_transfer_regime2_rf.csv')
print("\n=== Demo B regime 2 (Portmap=ddos): 2018 -> 2019(03-11) ===")
print(tbl[['macro_f1','weighted_f1','mcc','fp_rate','brier','ece']].round(4))

In [ ]:
import numpy as np, gc, warnings, pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from driftbench import config, compute_metrics, metrics_frame
warnings.filterwarnings('ignore')
R = Path('/content/drive/MyDrive/drift-conference/results')

Xa = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'features_core.parquet').astype('float32')
ya = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
Xb = pd.read_parquet(config.PROCESSED_DIR/'CIC-DDoS2019'/'features_core.parquet').astype('float32')
yb = pd.read_parquet(config.PROCESSED_DIR/'CIC-DDoS2019'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
sh=['benign','ddos']; ma=ya.isin(sh).values; mb=yb.isin(sh).values
Xa,ya=Xa[ma],ya[ma]; Xb,yb=Xb[mb].reset_index(drop=True)[Xa.columns],yb[mb].reset_index(drop=True)

base=RandomForestClassifier(n_estimators=120,max_depth=24,n_jobs=-1,random_state=0,class_weight='balanced_subsample').fit(Xa,ya)
rows={}; rows['zero_shot']=compute_metrics(yb,base.predict(Xb),base.predict_proba(Xb),base.classes_)
order=np.arange(len(Xb)); rng=np.random.default_rng(0); rng.shuffle(order)   # random buffer (matches regime 2)
for frac in (0.01,0.05,0.10):
    k=max(int(len(order)*frac),1); buf,rest=order[:k],order[k:]
    try:
        from sklearn.frozen import FrozenEstimator
        cal=CalibratedClassifierCV(FrozenEstimator(base),method='sigmoid')
    except ImportError:
        cal=CalibratedClassifierCV(base,method='sigmoid',cv='prefit')
    cal.fit(Xb.iloc[buf],yb.iloc[buf])
    rows[f'buffer_{int(frac*100)}pct']=compute_metrics(yb.iloc[rest],cal.predict(Xb.iloc[rest]),cal.predict_proba(Xb.iloc[rest]),cal.classes_)
    gc.collect()
tbl=metrics_frame(rows); tbl.to_csv(R/'demoB_transfer_rf.csv')
print("=== Demo B regime 1 (random buffer): 2018 -> 2019(01-12) ===")
print(tbl[['macro_f1','weighted_f1','mcc','fp_rate','brier','ece']].round(4))

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
R = Path('/content/drive/MyDrive/drift-conference/results'); F = R/'figures'; F.mkdir(exist_ok=True)
d1 = pd.read_csv(R/'demoB_transfer_rf.csv', index_col=0)
d2 = pd.read_csv(R/'demoB_transfer_regime2_rf.csv', index_col=0)
x=[0,1,5,10]
def col(d,m): return [d.loc['zero_shot',m],d.loc['buffer_1pct',m],d.loc['buffer_5pct',m],d.loc['buffer_10pct',m]]

fig, ax1 = plt.subplots(figsize=(7.2,4.4))
ax1.plot(x, col(d1,'ece'), 'o-', color='#C44E52', lw=2, ms=6, label='ECE — regime 1 (01-12)')
ax1.plot(x, col(d2,'ece'), 'o--', color='#E8896B', lw=2, ms=6, label='ECE — regime 2 (03-11)')
ax1.set_xlabel('Target calibration buffer (% of 2019)'); ax1.set_ylabel('Expected Calibration Error', color='#C44E52')
ax1.tick_params(axis='y', labelcolor='#C44E52'); ax1.set_ylim(0,1)
ax2 = ax1.twinx()
ax2.plot(x, col(d1,'mcc'), 's-', color='#4C72B0', lw=2, ms=6, label='MCC — regime 1')
ax2.plot(x, col(d2,'mcc'), 's--', color='#6B9BD1', lw=2, ms=6, label='MCC — regime 2')
ax2.set_ylabel('Matthews Correlation Coefficient', color='#4C72B0'); ax2.tick_params(axis='y', labelcolor='#4C72B0')
ax2.set_ylim(-0.15,1); ax2.axhline(0, color='#4C72B0', ls=':', alpha=0.4)
ax1.set_xticks(x); ax1.grid(alpha=0.3); ax1.set_axisbelow(True)
for s in ['top']: ax1.spines[s].set_visible(False); ax2.spines[s].set_visible(False)
h1,l1=ax1.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, frameon=False, fontsize=8, loc='center right')
plt.title('Cross-dataset transfer (two regimes): recalibration repairs confidence, not discrimination', fontsize=9.5)
plt.tight_layout()
fig.savefig(F/'fig_demoB_recovery.pdf'); fig.savefig(F/'fig_demoB_recovery.png', dpi=200)
plt.show(); print("saved updated fig_demoB_recovery (two regimes)")

In [ ]:
import pandas as pd
for ds in ['CIC-DDoS2019','CIC-DDoS2019-test']:
    m = pd.read_parquet(f'/content/drive/MyDrive/drift-conference/data/interim/{ds}/metadata.parquet')
    print(ds, '| cols:', list(m.columns))
    if 'timestamp' in m.columns:
        ts = pd.to_datetime(m['timestamp'], errors='coerce', dayfirst=True)
        print('   parseable:', ts.notna().mean(), '| range:', ts.min(), '->', ts.max(), '| unique days:', ts.dt.date.nunique())

In [ ]:
import pandas as pd, numpy as np, re, gc, warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from driftbench import config, compute_metrics, metrics_frame, harmonise_labels
warnings.filterwarnings('ignore')
R = Path('/content/drive/MyDrive/drift-conference/results')

def canon(col):
    c=col.strip().lower(); c=re.sub(r'[\s]+','_',c)
    for pat,sub in [(r'\bpkt\b','packet'),(r'\bpkts\b','packets'),(r'\blen\b','length'),
        (r'\bbyts\b','bytes'),(r'\bcnt\b','count'),(r'\btot\b','total'),
        (r'segment_size','seg_size'),(r'seg_size_avg','seg_size_mean'),
        (r'avg_(fwd|bwd)_seg_size',r'\1_seg_size_mean'),
        (r'init_win_bytes_forward','init_fwd_win_bytes'),(r'init_win_bytes_backward','init_bwd_win_bytes'),
        (r'act_data_pkt_fwd','fwd_act_data_packets'),(r'fwd_act_data_pkts','fwd_act_data_packets'),
        (r'_byts/s','_bytes/s'),(r'flow_byts/s','flow_bytes/s'),(r'flow_pkts/s','flow_packets/s'),
        (r'\bpacket_length\b','packet_len')]:
        c=re.sub(pat,sub,c)
    return c.replace('packet','pkt').replace('length','len')
core = pd.read_csv(config.PROCESSED_DIR/'common_core_features.csv').iloc[:,0].tolist()

# train on 2018 shared families (unchanged)
Xa = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'features_core.parquet').astype('float32')
ya = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
sh=['benign','ddos']; ma=ya.isin(sh).values; Xa,ya=Xa[ma],ya[ma]
base=RandomForestClassifier(n_estimators=120,max_depth=24,n_jobs=-1,random_state=0,class_weight='balanced_subsample').fit(Xa,ya)

def run_regime(interim_name, portmap_fix, out_csv):
    p = config.INTERIM_DIR/interim_name
    Xt = pd.read_parquet(p/'features.parquet').rename(columns=lambda c: canon(c))
    Xt = Xt.loc[:, [c for c in core if c in Xt.columns]].loc[:, lambda d: ~d.columns.duplicated()].astype('float32')
    yraw = pd.read_parquet(p/'labels.parquet')['label']
    yt = harmonise_labels(yraw).astype(str).reset_index(drop=True)
    if portmap_fix:
        yt = yt.mask(yraw.str.strip().str.lower().str.contains('portmap'), 'ddos').reset_index(drop=True)
    ts = pd.to_datetime(pd.read_parquet(p/'metadata.parquet')['timestamp'], dayfirst=True, errors='coerce')
    order = np.argsort(ts.values, kind='stable')   # CHRONOLOGICAL order, earliest first
    mt = yt.isin(sh).values
    Xt,yt = Xt[mt].reset_index(drop=True)[Xa.columns], yt[mt].reset_index(drop=True)
    order = order[np.isin(order, np.where(mt)[0])]  # restrict order to kept rows
    # remap order indices to post-filter positions
    pos = {old:i for i,old in enumerate(np.where(mt)[0])}
    order = np.array([pos[o] for o in order])

    rows={}
    rows['zero_shot']=compute_metrics(yt, base.predict(Xt), base.predict_proba(Xt), base.classes_)
    for frac in (0.01,0.05,0.10):
        k=max(int(len(order)*frac),1)
        buf, rest = order[:k], order[k:]          # earliest k% -> buffer; later -> test
        try:
            from sklearn.frozen import FrozenEstimator
            cal=CalibratedClassifierCV(FrozenEstimator(base),method='sigmoid')
        except ImportError:
            cal=CalibratedClassifierCV(base,method='sigmoid',cv='prefit')
        cal.fit(Xt.iloc[buf],yt.iloc[buf])
        rows[f'buffer_{int(frac*100)}pct']=compute_metrics(yt.iloc[rest],cal.predict(Xt.iloc[rest]),cal.predict_proba(Xt.iloc[rest]),cal.classes_)
        gc.collect()
    tbl=metrics_frame(rows); tbl.to_csv(R/out_csv)
    print(f"\n=== {interim_name} (TIME-ORDERED buffer) ===")
    print(tbl[['macro_f1','weighted_f1','mcc','fp_rate','brier','ece']].round(4))

run_regime('CIC-DDoS2019', False, 'demoB_transfer_rf.csv')
run_regime('CIC-DDoS2019-test', True, 'demoB_transfer_regime2_rf.csv')

In [ ]:
import pandas as pd
R='/content/drive/MyDrive/drift-conference/results/'
print(pd.read_csv(R+'demoB_transfer_rf.csv',index_col=0)[['macro_f1','mcc','fp_rate','brier','ece']].round(4))
print()
print(pd.read_csv(R+'demoB_transfer_regime2_rf.csv',index_col=0)[['macro_f1','mcc','fp_rate','brier','ece']].round(4))

In [ ]:
import pandas as pd, numpy as np, gc, warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score
from driftbench import config, random_split
warnings.filterwarnings('ignore')
R = Path('/content/drive/MyDrive/drift-conference/results')

p = config.INTERIM_DIR/'CSE-CIC-IDS2018'
X = pd.read_parquet(p/'features.parquet').astype('float32')
y = pd.read_parquet(p/'labels.parquet')['label'].reset_index(drop=True)
ts = pd.to_datetime(pd.read_parquet(p/'metadata.parquet')['timestamp'], dayfirst=True, errors='coerce')
order = np.argsort(ts.values, kind='stable')
classes = sorted(y.unique())                      # fixed class order for all protocols

def rf(): return RandomForestClassifier(n_estimators=120, max_depth=24, n_jobs=-1,
                                        random_state=0, class_weight='balanced_subsample')

def per_class_recall(tr, te):
    m = rf().fit(X.iloc[tr], y.iloc[tr])
    pred = m.predict(X.iloc[te]); ytrue = y.iloc[te]
    # recall per class over the FIXED class list; NaN where class absent from this test set
    rec = recall_score(ytrue, pred, labels=classes, average=None, zero_division=np.nan)
    train_classes = set(y.iloc[tr].unique())
    del m; gc.collect()
    return rec, train_classes

# build the three splits exactly as in Demo A
rs = random_split(len(X), y, seed=0)
tr_idx, te_idx = [], []
for cls in y.unique():
    idx_c = order[y.iloc[order].values == cls]; cut = int(len(idx_c)*0.6)
    tr_idx += list(idx_c[:cut]); te_idx += list(idx_c[cut:])
tr_idx, te_idx = np.array(tr_idx), np.array(te_idx)
a = int(len(X)*0.6); chrono_tr, chrono_te = order[:a], order[a:]

protocols = {'random': (rs['train'], rs['test']),
             'temporal_stratified': (tr_idx, te_idx),
             'chronological_blind': (chrono_tr, chrono_te)}

rows = {}
trainsets = {}
for name,(tr,te) in protocols.items():
    rec, tc = per_class_recall(tr, te)
    rows[name] = rec
    trainsets[name] = tc
    print(f"{name}: computed")

df = pd.DataFrame(rows, index=classes)
df.index.name = 'class'
# annotate which classes were ABSENT from training per protocol (recall structurally 0)
for name in protocols:
    df[name+'_in_train'] = [c in trainsets[name] for c in classes]
df.to_csv(R/'demoA_per_class_recall_rf.csv')
print("\nsaved demoA_per_class_recall_rf.csv\n")
print(df.round(3).to_string())

In [ ]:
print(type(base).__name__)
try:
    print(base.get_params())
except Exception as e:
    print("not a plain estimator:", e)

In [ ]:
import pandas as pd, numpy as np, re, gc, warnings
from pathlib import Path
from sklearn.calibration import CalibratedClassifierCV
from driftbench import config, compute_metrics, metrics_frame, harmonise_labels, make_model
warnings.filterwarnings('ignore')
R = Path('/content/drive/MyDrive/drift-conference/results')

def canon(col):
    c=col.strip().lower(); c=re.sub(r'[\s]+','_',c)
    for pat,sub in [(r'\bpkt\b','packet'),(r'\bpkts\b','packets'),(r'\blen\b','length'),
        (r'\bbyts\b','bytes'),(r'\bcnt\b','count'),(r'\btot\b','total'),
        (r'segment_size','seg_size'),(r'seg_size_avg','seg_size_mean'),
        (r'avg_(fwd|bwd)_seg_size',r'\1_seg_size_mean'),
        (r'init_win_bytes_forward','init_fwd_win_bytes'),(r'init_win_bytes_backward','init_bwd_win_bytes'),
        (r'act_data_pkt_fwd','fwd_act_data_packets'),(r'fwd_act_data_pkts','fwd_act_data_packets'),
        (r'_byts/s','_bytes/s'),(r'flow_byts/s','flow_bytes/s'),(r'flow_pkts/s','flow_packets/s'),
        (r'\bpacket_length\b','packet_len')]:
        c=re.sub(pat,sub,c)
    return c.replace('packet','pkt').replace('length','len')
core = pd.read_csv(config.PROCESSED_DIR/'common_core_features.csv').iloc[:,0].tolist()

# TRAIN on 2018 {benign,ddos} with the SAME model as Demo A: make_model('rf') = 300 trees, unbounded depth
Xa = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'features_core.parquet').astype('float32')
ya = pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
sh=['benign','ddos']; ma=ya.isin(sh).values; Xa,ya=Xa[ma],ya[ma]
base = make_model('rf', seed=0).fit(Xa, ya)         # <-- the only change vs before
print("RF params:", base.get_params()['n_estimators'], "trees, max_depth", base.get_params()['max_depth'])

def run_regime(interim_name, portmap_fix, out_csv):
    p = config.INTERIM_DIR/interim_name
    Xt = pd.read_parquet(p/'features.parquet').rename(columns=lambda c: canon(c))
    Xt = Xt.loc[:, [c for c in core if c in Xt.columns]].loc[:, lambda d: ~d.columns.duplicated()].astype('float32')
    yraw = pd.read_parquet(p/'labels.parquet')['label']
    yt = harmonise_labels(yraw).astype(str).reset_index(drop=True)
    if portmap_fix:
        yt = yt.mask(yraw.str.strip().str.lower().str.contains('portmap'), 'ddos').reset_index(drop=True)
    ts = pd.to_datetime(pd.read_parquet(p/'metadata.parquet')['timestamp'], dayfirst=True, errors='coerce')
    order = np.argsort(ts.values, kind='stable')
    mt = yt.isin(sh).values
    Xt,yt = Xt[mt].reset_index(drop=True)[Xa.columns], yt[mt].reset_index(drop=True)
    pos = {old:i for i,old in enumerate(np.where(mt)[0])}
    order = np.array([pos[o] for o in order if o in pos])

    rows={}
    rows['zero_shot']=compute_metrics(yt, base.predict(Xt), base.predict_proba(Xt), base.classes_)
    for frac in (0.01,0.05,0.10):
        k=max(int(len(order)*frac),1)
        buf, rest = order[:k], order[k:]
        try:
            from sklearn.frozen import FrozenEstimator
            cal=CalibratedClassifierCV(FrozenEstimator(base),method='sigmoid')
        except ImportError:
            cal=CalibratedClassifierCV(base,method='sigmoid',cv='prefit')
        cal.fit(Xt.iloc[buf],yt.iloc[buf])
        rows[f'buffer_{int(frac*100)}pct']=compute_metrics(yt.iloc[rest],cal.predict(Xt.iloc[rest]),cal.predict_proba(Xt.iloc[rest]),cal.classes_)
        gc.collect()
    tbl=metrics_frame(rows); tbl.to_csv(R/out_csv)
    print(f"\n=== {interim_name} (300-tree RF, time-ordered buffer) ===")
    print(tbl[['macro_f1','weighted_f1','mcc','auprc','fp_rate','brier','ece']].round(4))

run_regime('CIC-DDoS2019', False, 'demoB_transfer_rf.csv')
run_regime('CIC-DDoS2019-test', True, 'demoB_transfer_regime2_rf.csv')